<a href="https://colab.research.google.com/github/scarlettyu2023/AI_agent_workshop/blob/main/Topic6VLM/Topic_6_Vision_Language_Models_task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U "transformers>=4.39" "accelerate>=0.26" bitsandbytes langgraph gradio pillow

In [ ]:
!pip uninstall -y pillow
!pip install pillow==10.3.0

Found existing installation: pillow 12.1.1
Uninstalling pillow-12.1.1:
  Successfully uninstalled pillow-12.1.1
  Using cached pillow-10.3.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (9.2 kB)
Using cached pillow-10.3.0-cp312-cp312-manylinux_2_28_x86_64.whl (4.5 MB)


In [ ]:
!git config --global credential.helper store
!hf auth login

A new version of huggingface_hub (1.5.0) is available! You are using version 1.4.1.
To update, run: pip install -U huggingface_hub


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: y
Token is valid (permission: read).
The token `scarlettyucs6501` has been saved to /root/.cache/huggingfa

In [ ]:
import torch
from typing import TypedDict, List, Dict, Any, Optional

import gradio as gr
from PIL import Image

from langgraph.graph import StateGraph, START, END

from transformers import AutoTokenizer, CLIPImageProcessor, LlavaProcessor, LlavaForConditionalGeneration

MODEL_ID = "bczhou/tiny-llava-v1-hf"

def load_tiny_llava():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

    # Image processor (CLIP)
    image_processor = CLIPImageProcessor.from_pretrained(MODEL_ID)

    # Build LlavaProcessor manually
    processor = LlavaProcessor(tokenizer=tokenizer, image_processor=image_processor)

    # IMPORTANT: ensure patch_size is set (some tiny checkpoints miss it)
    # Most LLaVA CLIP backbones use patch_size=14.
    if getattr(processor, "patch_size", None) is None:
        processor.patch_size = 14

    model = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    if device != "cuda":
        model = model.to(device)

    model.eval()
    print("Model loaded:", MODEL_ID)
    return processor, model, device

processor, model, device = load_tiny_llava()

# ============================================================
# LangGraph State Definition
# ============================================================

class AgentState(TypedDict):
    messages: List[Dict[str, Any]]   # conversation history
    user_input: str                  # latest user message
    assistant_output: str            # latest model response
    image: Optional[Image.Image]     # uploaded image (persistent)


SYSTEM_PROMPT = (
    "You are a helpful vision-language assistant. "
    "Answer questions using the image and the conversation context. "
    "If something is not visible in the image, say you are not sure."
)

# ============================================================
# Prompt Builder
# ============================================================

def build_prompt_from_history(messages: List[Dict[str, Any]], user_text: str) -> str:
    """
    Build a compact text prompt from multi-turn history.
    LLaVA expects text + image input.
    """
    lines = []
    for m in messages:
        role = m["role"]
        content = m["content"]

        if role == "system":
            lines.append(f"[SYSTEM] {content}")
        elif role == "user":
            lines.append(f"User: {content}")
        elif role == "assistant":
            lines.append(f"Assistant: {content}")

    lines.append(f"User: {user_text}")
    lines.append("Assistant:")
    return "\n".join(lines)


# ============================================================
# LLaVA Inference
# ============================================================

def llava_generate(image: Image.Image, history: List[Dict[str, Any]], user_text: str, max_new_tokens: int = 128) -> str:
    """
    TinyLLaVA does not provide a chat template, so we manually format the prompt.
    IMPORTANT: include the <image> placeholder token so image tokens match features.
    """
    # Build a compact conversation prompt
    lines = []
    for m in history:
        role = m["role"]
        content = m["content"]

        if role == "system":
            # Keep system instruction minimal
            lines.append(f"SYSTEM: {content}")
        elif role == "user":
            lines.append(f"USER: {content}")
        elif role == "assistant":
            lines.append(f"ASSISTANT: {content}")

    # Current user turn MUST include <image>
    lines.append(f"USER: <image>\n{user_text}")
    lines.append("ASSISTANT:")

    prompt = "\n".join(lines)

    inputs = processor(text=prompt, images=image, return_tensors="pt")
    for k in inputs:
        inputs[k] = inputs[k].to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    decoded = processor.decode(output_ids[0], skip_special_tokens=True)

    # Extract only the assistant part (best-effort)
    if "ASSISTANT:" in decoded:
        decoded = decoded.split("ASSISTANT:")[-1].strip()

    return decoded.strip()


# ============================================================
# LangGraph Construction
# ============================================================

def create_graph():

    def call_vlm(state: AgentState) -> dict:
        user_text = (state["user_input"] or "").strip()
        if user_text == "":
            user_text = "Please describe the image."

        image = state.get("image", None)
        if image is None:
            return {"assistant_output": "Please upload an image first."}

        history = state["messages"]
        if not history or history[0].get("role") != "system":
            history = [{"role": "system", "content": SYSTEM_PROMPT}] + history

        answer = llava_generate(image, history, user_text, max_new_tokens=128)

        new_history = history + [
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": answer},
        ]

        return {
            "assistant_output": answer,
            "messages": new_history,
        }

    graph_builder = StateGraph(AgentState)
    graph_builder.add_node("call_vlm", call_vlm)
    graph_builder.add_edge(START, "call_vlm")
    graph_builder.add_edge("call_vlm", END)

    return graph_builder.compile()


graph = create_graph()

# ============================================================
# Initial State
# ============================================================

def init_state() -> AgentState:
    return {
        "messages": [{"role": "system", "content": SYSTEM_PROMPT}],
        "user_input": "",
        "assistant_output": "",
        "image": None,
    }

# ============================================================
# Gradio Chat Interface
# ============================================================

def on_chat(user_text, image, state, chat_history):
    if state is None or state == {}:
        state = init_state()

    # First turn requires image
    if state["image"] is None:
        if image is None:
            chat_history = chat_history or []
            chat_history.append(("System", "Please upload an image first."))
            return chat_history, state

        pil = image.convert("RGB")

        # Resize large images for speed
        max_side = 1024
        w, h = pil.size
        if max(w, h) > max_side:
            scale = max_side / float(max(w, h))
            pil = pil.resize((int(w * scale), int(h * scale)))

        state["image"] = pil

    state["user_input"] = user_text
    out = graph.invoke(state)
    state.update(out)

    chat_history = chat_history or []
    chat_history.append((user_text, state["assistant_output"]))
    return chat_history, state


def on_clear():
    return [], init_state()


with gr.Blocks() as demo:
    gr.Markdown("# Exercise 1 — Vision-Language LangGraph Chat Agent")

    with gr.Row():
        image_input = gr.Image(type="pil", label="Upload Image (required once)")
        state_store = gr.State(init_state())

    chatbot = gr.Chatbot(height=420)
    text_input = gr.Textbox(
        label="Your message",
        placeholder="Ask something about the image...",
        lines=2
    )

    with gr.Row():
        send_btn = gr.Button("Send")
        clear_btn = gr.Button("Clear")

    send_btn.click(
        on_chat,
        inputs=[text_input, image_input, state_store, chatbot],
        outputs=[chatbot, state_store],
    )

    text_input.submit(
        on_chat,
        inputs=[text_input, image_input, state_store, chatbot],
        outputs=[chatbot, state_store],
    )

    clear_btn.click(on_clear, inputs=None, outputs=[chatbot, state_store])

demo.queue().launch(share=True, debug=True, show_error=True)

Device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/596 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Model loaded: bczhou/tiny-llava-v1-hf


/tmp/ipython-input-1291/160647819.py:238: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=420)
/tmp/ipython-input-1291/160647819.py:238: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=420)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://81e98ab976547b572d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
